# 6 · Tuples, Sets and Dictionaries

*Built-in data structures* · 10 exercises

Hashing structures that turn searches into O(1) lookups.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-6) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-6)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 6.1 · Tuples and unpacking  `Core`

**Task.** Write `min_max(arr)` returning a tuple, then `head_tail(arr)` using star unpacking (`first, *rest = arr`). Also swap two variables without a temporary.

**Example.** `min_max([4, 1, 9])` -> `(1, 9)`; `head_tail([4, 1, 9])` -> `(4, [1, 9])`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-6-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-6-1)

In [ ]:
# 6.1 Tuples and unpacking
def min_max(arr):
    raise NotImplementedError


def head_tail(arr):
    raise NotImplementedError

In [ ]:
def test_ex_6_1():
    assert min_max([4, 1, 9]) == (1, 9)
    assert head_tail([4, 1, 9]) == (4, [1, 9])


test_ex_6_1()
print('6.1 passed')

### 6.2 · Set operations  `Core`

**Task.** Write `compare(a, b)` returning `(common, only_a, either_not_both)` as sorted lists, using `&`, `-` and `^`.

**Example.** `compare([1, 2, 3, 4], [3, 4, 5])` -> `([3, 4], [1, 2], [1, 2, 5])`

**Big O.** O(N + M).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-6-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-6-2)

In [ ]:
# 6.2 Set operations
def compare(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_6_2():
    assert compare([1, 2, 3, 4], [3, 4, 5]) == ([3, 4], [1, 2], [1, 2, 5])


test_ex_6_2()
print('6.2 passed')

### 6.3 · Count with a dict  `Core`

**Task.** Write `word_counts(text)` returning a dict of lowercase word to count, using `d[w] = d.get(w, 0) + 1`.

**Example.** `word_counts("To be or not to be")` -> `{'to': 2, 'be': 2, 'or': 1, 'not': 1}`

**Big O.** O(N) words.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-6-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-6-3)

In [ ]:
# 6.3 Count with a dict
def word_counts(text):
    raise NotImplementedError

In [ ]:
def test_ex_6_3():
    assert word_counts('To be or not to be') == {'to': 2, 'be': 2, 'or': 1, 'not': 1}


test_ex_6_3()
print('6.3 passed')

### 6.4 · Invert a dictionary  `Core`

**Task.** Write `invert(d)` that swaps keys and values, assuming values are unique, with a dict comprehension.

**Example.** `invert({'a': 1, 'b': 2})` -> `{1: 'a', 2: 'b'}`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-6-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-6-4)

In [ ]:
# 6.4 Invert a dictionary
def invert(d):
    raise NotImplementedError

In [ ]:
def test_ex_6_4():
    assert invert({'a': 1, 'b': 2}) == {1: 'a', 2: 'b'}


test_ex_6_4()
print('6.4 passed')

### 6.5 · Group with setdefault  `Core`

**Task.** Write `group_by_first(words)` mapping each first letter to the list of words that start with it, using `setdefault`.

**Example.** `group_by_first(["apple", "bat", "avocado"])` -> `{'a': ['apple', 'avocado'], 'b': ['bat']}`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-6-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-6-5)

In [ ]:
# 6.5 Group with setdefault
def group_by_first(words):
    raise NotImplementedError

In [ ]:
def test_ex_6_5():
    assert group_by_first(['apple', 'bat', 'avocado']) == {'a': ['apple', 'avocado'], 'b': ['bat']}


test_ex_6_5()
print('6.5 passed')

### 6.6 · Sort a dict by value, then key  `Core`

**Task.** Write `top_words(counts, k)` returning the k most frequent words, ties broken alphabetically, using `sorted(counts.items(), key=...)`.

**Example.** `top_words({'b': 2, 'a': 2, 'c': 5}, 2)` -> `['c', 'a']`

**Big O.** O(N log N).

**Note.** A tuple key `(-count, word)` sorts by count descending and word ascending in one pass.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-6-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-6-6)

In [ ]:
# 6.6 Sort a dict by value, then key
def top_words(counts, k):
    raise NotImplementedError

In [ ]:
def test_ex_6_6():
    assert top_words({'b': 2, 'a': 2, 'c': 5}, 2) == ['c', 'a']


test_ex_6_6()
print('6.6 passed')

### 6.7 · Tuples as dict keys  `Core`

**Task.** Write `busiest_cell(points)` returning the `(x, y)` that appears most often in a list of points, using tuples as dictionary keys. Break ties by the smallest point.

**Example.** `busiest_cell([(1, 2), (0, 0), (1, 2), (0, 0), (3, 3)])` -> `(0, 0)`

**Big O.** O(N).

**Note.** Lists cannot be dict keys or set members because they are mutable; tuples can.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-6-7) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-6-7)

In [ ]:
# 6.7 Tuples as dict keys
def busiest_cell(points):
    raise NotImplementedError

In [ ]:
def test_ex_6_7():
    assert busiest_cell([(1, 2), (0, 0), (1, 2), (0, 0), (3, 3)]) == (0, 0)


test_ex_6_7()
print('6.7 passed')

### 6.8 · Unique pairs with a target sum  `Practice`

**Task.** Write `pairs_with_sum(nums, target)` returning the set of unordered pairs `(small, large)` that add to target.

**Example.** `pairs_with_sum([1, 5, 3, 3, 2, 4], 6)` -> `{(1, 5), (3, 3), (2, 4)}`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-6-8) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-6-8)

In [ ]:
# 6.8 Unique pairs with a target sum
def pairs_with_sum(nums, target):
    raise NotImplementedError

In [ ]:
def test_ex_6_8():
    assert pairs_with_sum([1, 5, 3, 3, 2, 4], 6) == {(1, 5), (3, 3), (2, 4)}
    assert pairs_with_sum([3], 6) == set()


test_ex_6_8()
print('6.8 passed')

### 6.9 · Dict and set comprehensions  `Core`

**Task.** Write `squares_map(n)` returning `{i: i*i}` for 1..n, and `remainders(nums, k)` returning the set of distinct `x % k`.

**Example.** `squares_map(3)` -> `{1: 1, 2: 4, 3: 9}`; `remainders([5, 7, 9, 10], 3)` -> `{0, 1, 2}`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-6-9) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-6-9)

In [ ]:
# 6.9 Dict and set comprehensions
def squares_map(n):
    raise NotImplementedError


def remainders(nums, k):
    raise NotImplementedError

In [ ]:
def test_ex_6_9():
    assert squares_map(3) == {1: 1, 2: 4, 3: 9}
    assert remainders([5, 7, 9, 10], 3) == {0, 1, 2}


test_ex_6_9()
print('6.9 passed')

### 6.10 · Merge two count dicts  `Practice`

**Task.** Write `merge_counts(a, b)` returning a new dict where shared keys add their values. Do not modify the inputs.

**Example.** `merge_counts({'x': 1, 'y': 2}, {'y': 5, 'z': 1})` -> `{'x': 1, 'y': 7, 'z': 1}`

**Big O.** O(N + M).

**Note.** `a | b` (Python 3.9) merges too, but the right side wins on shared keys instead of adding.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-6-10) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-6-10)

In [ ]:
# 6.10 Merge two count dicts
def merge_counts(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_6_10():
    a, b = ({'x': 1, 'y': 2}, {'y': 5, 'z': 1})
    assert merge_counts(a, b) == {'x': 1, 'y': 7, 'z': 1}
    assert a == {'x': 1, 'y': 2}


test_ex_6_10()
print('6.10 passed')